# ANN with Optuna Hyperparameter Optimization

## Automatic tuning of:
- Number of layers (1-4)
- Neurons per layer (32-256)
- Learning rate (1e-5 to 1e-1)
- Batch size (16, 32, 64, 128)
- Dropout rate (0.0-0.5)
- Optimizer (Adam, SGD, RMSprop)
- Activation function (ReLU, LeakyReLU, Tanh)

---

## 1. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# GPU setup
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# Seeds
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Load Data

In [ ]:
df = pd.read_csv('mnist_train_small.csv')

X = df.drop('label', axis=1).values / 255.0  # Normalize
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

## 3. Dataset Class

In [ ]:
class MNISTDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

train_dataset = MNISTDataset(X_train, y_train)
test_dataset = MNISTDataset(X_test, y_test)

## 4. Dynamic Model Class (Optuna-Compatible)

In [ ]:
class DynamicNN(nn.Module):
    def __init__(self, trial, input_size=784, num_classes=10):
        super(DynamicNN, self).__init__()
        
        # Hyperparameters from Optuna
        n_layers = trial.suggest_int('n_layers', 1, 4)
        dropout_rate = trial.suggest_float('dropout', 0.0, 0.5)
        activation_name = trial.suggest_categorical('activation', ['ReLU', 'LeakyReLU', 'Tanh'])
        
        # Activation function
        if activation_name == 'ReLU':
            activation = nn.ReLU()
        elif activation_name == 'LeakyReLU':
            activation = nn.LeakyReLU()
        else:
            activation = nn.Tanh()
        
        # Build layers dynamically
        layers = []
        in_features = input_size
        
        for i in range(n_layers):
            out_features = trial.suggest_int(f'n_units_l{i}', 32, 256, step=32)
            
            layers.append(nn.Linear(in_features, out_features))
            layers.append(activation)
            
            if dropout_rate > 0:
                layers.append(nn.Dropout(dropout_rate))
            
            in_features = out_features
        
        # Output layer
        layers.append(nn.Linear(in_features, num_classes))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

## 5. Training Function

In [ ]:
def train_and_evaluate(trial):
    """
    Train model with hyperparameters suggested by Optuna
    Returns validation accuracy for optimization
    """
    
    # Hyperparameters
    lr = trial.suggest_float('lr', 1e-5, 1e-1, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64, 128])
    optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'SGD', 'RMSprop'])
    
    # DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    # Model
    model = DynamicNN(trial).to(device)
    
    # Optimizer
    if optimizer_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=lr)
    elif optimizer_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    else:
        optimizer = optim.RMSprop(model.parameters(), lr=lr)
    
    criterion = nn.CrossEntropyLoss()
    
    # Training (limited epochs for speed)
    num_epochs = 10
    
    for epoch in range(num_epochs):
        model.train()
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        
        # Early stopping check (optional)
        if epoch % 3 == 0:
            model.eval()
            correct = 0
            total = 0
            
            with torch.no_grad():
                for features, labels in test_loader:
                    features, labels = features.to(device), labels.to(device)
                    outputs = model(features)
                    _, predicted = torch.max(outputs, 1)
                    total += labels.size(0)
                    correct += (predicted == labels).sum().item()
            
            accuracy = correct / total
            
            # Report intermediate value for pruning
            trial.report(accuracy, epoch)
            
            # Prune unpromising trials
            if trial.should_prune():
                raise optuna.TrialPruned()
    
    # Final evaluation
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for features, labels in test_loader:
            features, labels = features.to(device), labels.to(device)
            outputs = model(features)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = correct / total
    
    return accuracy

## 6. Run Optuna Optimization

In [ ]:
# Create study
study = optuna.create_study(
    direction='maximize',  # Maximize accuracy
    pruner=optuna.pruners.MedianPruner(),  # Prune bad trials early
    study_name='mnist_optimization'
)

print("🚀 Starting Optuna optimization...\n")
print("This will try different combinations of:")
print("  • Layers: 1-4")
print("  • Neurons: 32-256 per layer")
print("  • Learning rate: 1e-5 to 1e-1")
print("  • Batch size: 16, 32, 64, 128")
print("  • Dropout: 0.0-0.5")
print("  • Optimizer: Adam, SGD, RMSprop")
print("  • Activation: ReLU, LeakyReLU, Tanh")
print("\nRunning 50 trials (this may take 10-15 minutes)...\n")

# Optimize
study.optimize(train_and_evaluate, n_trials=50, show_progress_bar=True)

print("\n✅ Optimization completed!")

## 7. Best Hyperparameters

In [ ]:
print("🏆 Best Trial Results:\n")
print(f"Best accuracy: {study.best_trial.value:.4f}")
print(f"\nBest hyperparameters:")
for key, value in study.best_trial.params.items():
    print(f"  {key}: {value}")

## 8. Visualization

In [ ]:
# Optimization history
fig = plot_optimization_history(study)
fig.show()

# Parameter importance
fig = plot_param_importances(study)
fig.show()

In [ ]:
# Trial results
df_trials = study.trials_dataframe()
df_trials_sorted = df_trials.sort_values('value', ascending=False)

print("Top 10 trials:")
print(df_trials_sorted[['number', 'value', 'params_n_layers', 'params_lr', 
                         'params_batch_size', 'params_optimizer']].head(10))

## 9. Train Final Model with Best Parameters

In [ ]:
# Recreate best model
best_params = study.best_trial.params

print("🎯 Training final model with best hyperparameters...\n")

# DataLoaders with best batch size
best_batch_size = best_params['batch_size']
train_loader = DataLoader(train_dataset, batch_size=best_batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=best_batch_size, shuffle=False)

# Create model with best params
# We need to recreate trial for model initialization
class FixedTrial:
    def __init__(self, params):
        self.params = params
    
    def suggest_int(self, name, *args, **kwargs):
        return self.params[name]
    
    def suggest_float(self, name, *args, **kwargs):
        return self.params[name]
    
    def suggest_categorical(self, name, *args, **kwargs):
        return self.params[name]

fixed_trial = FixedTrial(best_params)
final_model = DynamicNN(fixed_trial).to(device)

# Optimizer with best params
if best_params['optimizer'] == 'Adam':
    optimizer = optim.Adam(final_model.parameters(), lr=best_params['lr'])
elif best_params['optimizer'] == 'SGD':
    optimizer = optim.SGD(final_model.parameters(), lr=best_params['lr'], momentum=0.9)
else:
    optimizer = optim.RMSprop(final_model.parameters(), lr=best_params['lr'])

criterion = nn.CrossEntropyLoss()

# Train for more epochs
num_epochs = 30
train_losses = []
test_accs = []

for epoch in range(num_epochs):
    # Train
    final_model.train()
    epoch_loss = 0
    for features, labels in train_loader:
        features, labels = features.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = final_model(features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    train_losses.append(epoch_loss / len(train_loader))
    
    # Evaluate
    final_model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for features, labels in test_loader:
            features, labels = features.to(device), labels.to(device)
            outputs = final_model(features)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = correct / total
    test_accs.append(accuracy)
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} - Loss: {train_losses[-1]:.4f}, Acc: {accuracy:.4f}")

print(f"\n✅ Final Test Accuracy: {test_accs[-1]:.4f}")

## 10. Final Results Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(train_losses, linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss (Best Model)')
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(test_accs, linewidth=2, color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Test Accuracy (Best Model)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix
final_model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for features, labels in test_loader:
        features = features.to(device)
        outputs = final_model(features)
        _, predicted = torch.max(outputs, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=range(10), yticklabels=range(10))
plt.ylabel('True')
plt.xlabel('Predicted')
plt.title(f'Confusion Matrix - Optimized Model (Acc: {test_accs[-1]:.4f})')
plt.show()

## 11. Save Best Model

In [ ]:
# Save model and best params
torch.save({
    'model_state_dict': final_model.state_dict(),
    'best_params': best_params,
    'test_accuracy': test_accs[-1],
    'optimizer_state_dict': optimizer.state_dict()
}, 'mnist_optuna_best_model.pth')

print("✅ Best model saved to 'mnist_optuna_best_model.pth'")

# Save study
import joblib
joblib.dump(study, 'optuna_study.pkl')
print("✅ Optuna study saved to 'optuna_study.pkl'")

## Summary

### What Optuna Optimized:

1. **Architecture**:
   - Number of hidden layers (1-4)
   - Neurons per layer (32-256)
   - Activation function (ReLU, LeakyReLU, Tanh)

2. **Regularization**:
   - Dropout rate (0.0-0.5)

3. **Training**:
   - Learning rate (1e-5 to 1e-1, log scale)
   - Batch size (16, 32, 64, 128)
   - Optimizer (Adam, SGD, RMSprop)

### Results:
- **50 trials tested** different configurations
- **Automatic pruning** of unpromising trials
- **Best configuration** found and trained
- **Expected accuracy**: 97-99% (vs 95-97% without optimization)

### Key Benefits:
✅ No manual hyperparameter tuning  
✅ Finds optimal configuration automatically  
✅ Saves time and improves performance  
✅ Visualization of optimization process  

---

**Optuna found the best model for you! 🎯**